# GreekBarRetrieval: paper experiments

This notebook is a compact, self-contained implementation of the **reported** experiments in [GreekBarRetrieval](https://arxiv.org/abs/2608.18752): three Greek BM25 variants, nine dense retrievers, English translation baselines, one-shot reformulation, BM25 tuning, sparse and dense PRF, reciprocal-rank fusion, and ten-round ReAct-BM25. It follows [the official retrieval quickstart](https://github.com/nlpaueb/greek-bar-bench/blob/main/quickstart/quickstart_greekbarretrieval.ipynb): load the Hugging Face `corpus`, `queries`, and `qrels` subsets, run retrieval, and evaluate against article-level qrels. It does not use qrels during retrieval, reformulation, or feedback.

**Navigation:** 1. Setup and data · 2. Metrics · 3. BM25 · 4. Dense · 5. Reformulation · 6. PRF, RRF, tuning, translation · 7. ReAct-BM25 · 8. Results and export.

Hugging Face `queries.text` already contains the facts and question in the benchmark order; the notebook uses it verbatim. Rankings contain at most 100 article IDs. Metrics are macro-averaged nDCG@10/100, Recall@10/100, and MAP@100. The experiment switches below prevent accidental multi-hour model runs. Set each switch to `True` to execute that family. For a quick inspection, set `QUERY_LIMIT` to a small number; full-dataset comparisons require `QUERY_LIMIT=None`.

Dataset source: [`AUEB-NLP/greek-bar-bench`](https://huggingface.co/datasets/AUEB-NLP/greek-bar-bench), revision `v2.0`. No local corpus or query files are read. The original study used locally served `gpt-oss-120b` and embedding models; configure the endpoints below before enabling those runs.

Switch dependencies: keep `RUN_SPARSE=True` for reformulation, PRF/tuning, and ReAct. Keep `RUN_DENSE=True` for PRF/RRF. Running all reported methods requires enabling all five method switches; significance testing is a separate optional switch at the end.

## 1. Setup and data

Install as needed (in a notebook environment): `pip install "datasets>=4.0" numpy pandas "rank_bm25==0.2.2" "greek-stemmer-plus==0.2.0" spacy gr-nlp-toolkit google-genai tiktoken`; then `python -m spacy download el_core_news_sm` and `python -m spacy download en_core_web_sm`. The maintained stemmer fork matches the official quickstart and works with current PyYAML. For local embedding and generation servers, expose OpenAI-compatible `/v1/embeddings` and `/v1/chat/completions`. Gemini uses `GOOGLE_API_KEY`. Pin the remaining package versions and model checkpoints for a final public release.

In [ ]:
from __future__ import annotations
import ast, csv, hashlib, json, math, os, re, time, unicodedata
from collections import Counter
from pathlib import Path
from urllib.request import Request, urlopen
import numpy as np
import pandas as pd

from datasets import load_dataset
REPO = "AUEB-NLP/greek-bar-bench"
REVISION = "v2.0"
OUTPUT_DIR = Path(os.getenv("GBR_OUTPUT_DIR", "./greekbarretrieval_runs")).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QUERY_LIMIT = None  # None = every query in the selected dataset revision
RUN_SPARSE = True
RUN_DENSE = False
RUN_REFORMULATION = False
RUN_PRF_RRF_TUNING = False
RUN_TRANSLATION = False
RUN_REACT = False

LMSTUDIO_URL = os.getenv("LMSTUDIO_EMBEDDING_BASE_URL", "http://localhost:1234/v1").rstrip("/")
CUSTOM_URL = os.getenv("CUSTOM_EMBEDDING_BASE_URL", "http://localhost:8000/v1").rstrip("/")
CHAT_URL = os.getenv("LOCAL_LLM_BASE_URL", LMSTUDIO_URL).rstrip("/")
CHAT_MODEL = os.getenv("LOCAL_LLM_MODEL", "openai/gpt-oss-120b")
EMBEDDING_API_KEY = os.getenv("EMBEDDING_API_KEY", "")
CHAT_API_KEY = os.getenv("LOCAL_LLM_API_KEY", "lm-studio")
print("Hugging Face dataset:", REPO, REVISION)

In [ ]:
ARTIFACTS = re.compile(r"NNBSP|NBSP|NARROW NO-BREAK SPACE|\\u202f|\\u00a0|\\xa0|[\u202f\u00a0\u2009\u200a\u200b\ufeff]")
def clean(value):
    value = ARTIFACTS.sub(" ", str(value or "")).replace("\r\n", "\n").replace("\r", "\n")
    return re.sub(r"[ \t]{2,}", " ", value).strip()

def load_benchmark():
    corpus = load_dataset(REPO, "corpus", split="test", revision=REVISION)
    query_set = load_dataset(REPO, "queries", split="test", revision=REVISION)
    qrel_set = load_dataset(REPO, "qrels", split="test", revision=REVISION)
    docs = dict(zip(corpus["id"], corpus["text"]))
    queries = dict(zip(query_set["id"], query_set["text"]))
    qrels = {qid: {} for qid in queries}
    for row in qrel_set:
        if int(row["score"]) > 0:
            qrels[row["query-id"]][row["corpus-id"]] = 1
    assert len(docs) == len(corpus) and len(queries) == len(query_set), "Duplicate benchmark IDs"
    assert all(set(gold) <= set(docs) for gold in qrels.values()), "Qrel ID missing from corpus"
    assert all(qrels.values()), "A query has no relevance judgments"
    if QUERY_LIMIT is not None:
        queries = dict(list(queries.items())[:QUERY_LIMIT]); qrels = {q: qrels[q] for q in queries}
    if not queries: raise ValueError("No queries were loaded")
    return docs, queries, qrels

docs, queries, qrels = load_benchmark()
print(f"Greek: {len(docs):,} articles; {len(queries):,} queries; {sum(map(len, qrels.values())):,} qrels")


## 2. Evaluation and run artifacts

All experiments use the same qrels and macro averages. A run is a mapping from query ID to a ranked list of document IDs. Artifacts are saved in JSON and TREC format for independent evaluation.

In [ ]:
def score_ranking(ranked, gold):
    gold = set(gold); ranked = list(dict.fromkeys(ranked))[:100]
    out = {}
    for k in (10, 100):
        top = ranked[:k]
        hits = sum(d in gold for d in top)
        out[f"Recall@{k}"] = hits / len(gold) if gold else 0.0
        dcg = sum((d in gold) / math.log2(i + 2) for i, d in enumerate(top))
        ideal = sum(1 / math.log2(i + 2) for i in range(min(k, len(gold))))
        out[f"nDCG@{k}"] = dcg / ideal if ideal else 0.0
    hits = 0; total = 0.0
    for i, d in enumerate(ranked, 1):
        if d in gold: hits += 1; total += hits / i
    out["MAP@100"] = total / min(100, len(gold)) if gold else 0.0
    return out

runs, summaries = {}, []
def register(name, ranking, *, details=None):
    if set(ranking) != set(queries): raise ValueError(f"{name}: missing query IDs")
    rankings = {q: list(dict.fromkeys(ranking[q]))[:100] for q in queries}
    per_query = pd.DataFrame({q: score_ranking(rankings[q], qrels[q]) for q in queries}).T
    row = {"experiment": name, **per_query.mean().to_dict()}
    summaries.append(row); runs[name] = rankings
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", name)
    (OUTPUT_DIR / f"{safe}.json").write_text(json.dumps(rankings, ensure_ascii=False, indent=2), encoding="utf-8")
    with (OUTPUT_DIR / f"{safe}.trec").open("w", encoding="utf-8") as f:
        for qid, ids in rankings.items():
            for rank, docid in enumerate(ids, 1):
                f.write(f"{qid} Q0 {docid} {rank} {101-rank} {safe}\n")
    if details is not None:
        (OUTPUT_DIR / f"{safe}.details.json").write_text(json.dumps(details, ensure_ascii=False, indent=2), encoding="utf-8")
    return row

def top_ids(scores, ids, k=100, positive_only=False):
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="stable")
    return [ids[i] for i in order if np.isfinite(scores[i]) and (not positive_only or scores[i] > 0)][:k]

## 3. Greek BM25 baselines

The paper uses `rank_bm25.BM25Okapi` with `k1=1.5`, `b=0.75`. GreekStemmer tokenization removes diacritics, uppercases, removes the original short stopword list, then stems. spaCy uses Greek lemmas and stopwords. gr-nlp-toolkit uses its tokenizer without stemming or case normalization. The latter two are included because they are reported in Appendix C.

In [ ]:
from rank_bm25 import BM25Okapi
from greek_stemmer_plus import GreekStemmer
STOP = set("ΚΑΙ Η Ο ΤΟ ΤΑ ΤΟΥ ΤΗΣ ΤΩΝ ΣΤΟ ΣΤΗ ΣΤΗΝ ΣΤΑ ΣΕ ΜΕ ΓΙΑ ΑΠΟ ΩΣ ΠΟΥ ΟΤΙ ΟΤΑΝ ΑΝ ΝΑ ΔΕΝ ΜΗ ΜΗΝ ΤΙ ΠΩΣ ΠΡΟΣ ΥΠΟ ΚΑΤΑ ΕΝ ΕΠΙ ΜΕΤΑ ΠΡΙΝ ΠΑΝΩ ΚΑΤΩ ΕΩΣ ΗΤΑΝ ΕΙΝΑΙ ΕΙΜΑΙ ΕΧΕΙ ΕΧΟΥΝ ΕΧΩ".split())
stemmer = GreekStemmer()
def strip_marks(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if not unicodedata.combining(c))
def greekstem_tokens(s):
    words = re.findall(r"\w+", clean(s), flags=re.UNICODE)
    norm = [strip_marks(x).upper() for x in words]
    return [stemmer.stem(x) for x in norm if x and x not in STOP]

def make_spacy_tokens(language="el"):
    import spacy
    from spacy.lang.el.stop_words import STOP_WORDS as EL_STOP
    from spacy.lang.en.stop_words import STOP_WORDS as EN_STOP
    nlp = spacy.load("el_core_news_sm" if language == "el" else "en_core_web_sm")
    nlp.max_length = max(nlp.max_length, 2_000_000)
    stop = {strip_marks(x).lower() for x in (EL_STOP if language == "el" else EN_STOP)}
    def tokenize(s):
        out = []
        for t in nlp(clean(s)):
            if t.is_space or t.is_punct: continue
            token = strip_marks(t.lemma_ or t.text).lower().strip() if language == "el" else (t.lemma_ or t.text).lower().strip()
            if token and token not in stop: out.append(token)
        return out
    return tokenize

def make_grnlp_tokens():
    from gr_nlp_toolkit.domain.document import Document
    from gr_nlp_toolkit.processors.tokenizer import Tokenizer
    tokenizer = Tokenizer()
    punct = set("!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~·΄—«»“”…‚„‘’–؛،")
    def tokenize(s):
        out = []
        for i in range(0, len(s), 2000):
            out.extend(clean(t.text) for t in tokenizer(Document(clean(s[i:i+2000]))).tokens)
        return [t for t in out if t and t not in punct]
    return tokenize

class SparseIndex:
    def __init__(self, corpus, tokenize, k1=1.5, b=0.75):
        self.ids = list(corpus); self.tokenize = tokenize
        self.tokens = [tokenize(corpus[d]) for d in self.ids]
        self.index = BM25Okapi(self.tokens, k1=k1, b=b)
    def search_tokens(self, tokens, k=100, positive_only=False):
        return top_ids(self.index.get_scores(tokens), self.ids, k, positive_only=positive_only)
    def search(self, text, k=100, positive_only=False):
        return self.search_tokens(self.tokenize(text), k, positive_only=positive_only)

sparse = {}
if RUN_SPARSE:
    sparse["BM25-GreekStemmer"] = SparseIndex(docs, greekstem_tokens)
    sparse["BM25-spaCy"] = SparseIndex(docs, make_spacy_tokens("el"))
    sparse["BM25-gr-nlp-toolkit"] = SparseIndex(docs, make_grnlp_tokens())
    for name, index in sparse.items():
        register(name, {q: index.search(text) for q, text in queries.items()})
    display(pd.DataFrame(summaries).round(3))

## 4. Dense retrieval: nine reported models

Eight local checkpoints use their paper-era HTTP endpoints and model-specific query/document prefixes. Gemini-001 uses Google's retrieval task types and 3,072 dimensions. Every article is encoded as one passage; vectors are L2-normalized and searched exactly by dot product. `embed_many` caches by corpus/query content and model formatting, so expensive article embeddings are reused.

In [ ]:
QWEN_TASK = "Given a legal question with facts, retrieve relevant legal passages, statutes, or documents that answer the query"
MODELS = {
    "Qwen3-8B": ("text-embedding-qwen3-embedding-8b", LMSTUDIO_URL, "Instruct: {instruction}\nQuery:{text}", "{text}"),
    "Qwen3-4B": ("text-embedding-qwen3-embedding-4b", LMSTUDIO_URL, "Instruct: {instruction}\nQuery:{text}", "{text}"),
    "Qwen3-0.6B": ("text-embedding-qwen3-embedding-0.6b", LMSTUDIO_URL, "Instruct: {instruction}\nQuery:{text}", "{text}"),
    "Euler-Legal-V1": ("Mira190/Euler-Legal-Embedding-V1", CUSTOM_URL, "{text}", "{text}"),
    "Jina-v5-small": ("jinaai/jina-embeddings-v5-text-small", CUSTOM_URL, "Query: {text}", "Document: {text}"),
    "Arctic-v2": ("Snowflake/snowflake-arctic-embed-l-v2.0", CUSTOM_URL, "query: {text}", "{text}"),
    "EmbGemma-300M": ("text-embedding-embeddinggemma-300m-qat", LMSTUDIO_URL, "task: search result | query: {text}", "title: none | text: {text}"),
    "Nomic-v1.5": ("text-embedding-nomic-embed-text-v1.5", LMSTUDIO_URL, "search_query: {text}", "search_document: {text}"),
    "Gemini-001": ("gemini-embedding-001", "google", "{text}", "{text}"),
}
def post_json(url, payload, key="", timeout=600):
    headers = {"Content-Type": "application/json"}
    if key: headers["Authorization"] = f"Bearer {key}"
    request = Request(url, data=json.dumps(payload).encode("utf-8"), headers=headers)
    with urlopen(request, timeout=timeout) as response:
        return json.load(response)

def embed_one(name, text, side):
    model, endpoint, qt, dt = MODELS[name]
    formatted = (qt if side == "query" else dt).format(text=text, instruction=QWEN_TASK)
    if endpoint == "google":
        from google import genai
        from google.genai import types
        client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
        kind = "RETRIEVAL_QUERY" if side == "query" else "RETRIEVAL_DOCUMENT"
        result = client.models.embed_content(model=model, contents=formatted, config=types.EmbedContentConfig(task_type=kind, output_dimensionality=3072))
        return np.asarray(result.embeddings[0].values, dtype=np.float32)
    response = post_json(endpoint + "/embeddings", {"model": model, "input": formatted}, EMBEDDING_API_KEY)
    return np.asarray(response["data"][0]["embedding"], dtype=np.float32)

def embed_many(name, texts, side, cache_tag):
    fingerprint = hashlib.sha256(json.dumps([name, MODELS[name], side, list(texts)], ensure_ascii=False).encode()).hexdigest()[:20]
    path = OUTPUT_DIR / f"emb_{cache_tag}_{fingerprint}.npy"
    if path.exists(): return np.load(path)
    array = np.vstack([embed_one(name, t, side) for t in texts]).astype(np.float32)
    array /= np.maximum(np.linalg.norm(array, axis=1, keepdims=True), 1e-12)
    np.save(path, array)
    return array

doc_ids = list(docs); query_ids = list(queries)
dense_vectors, dense_runs = {}, {}
def dense_run(name, query_texts=queries, corpus=docs, language="el"):
    ids = list(corpus); qids = list(query_texts)
    D = embed_many(name, list(corpus.values()), "document", f"{language}_docs")
    Q = embed_many(name, list(query_texts.values()), "query", f"{language}_queries")
    ranked = {q: top_ids(D @ Q[i], ids) for i, q in enumerate(qids)}
    return ranked, D, Q

if RUN_DENSE:
    for name in MODELS:
        run, D, Q = dense_run(name)
        dense_runs[name] = run; dense_vectors[name] = (D, Q)
        register(name, run)
    display(pd.DataFrame(summaries).round(3))

## 5. One-shot LLM query reformulation

The same `gpt-oss-120b` generates Greek legal keywords for sparse retrieval and concise Greek legal prose for dense retrieval. The prompt texts below are copied from the experiment scripts. Save generated rewrites to reuse them; their outputs may vary with the model server, seed, and sampling implementation.

In [ ]:
SPARSE_REWRITE_PROMPT = """Διάβασε το παρακάτω ελληνικό νομικό ερώτημα.

Ο ρόλος σου είναι να μετατρέψεις το αρχικό ερώτημα σε στοχευμένη λίστα λέξεων-κλειδιών και σύντομων νομικών φράσεων για sparse retrieval σε ελληνικά νομικά κείμενα. Εντόπισε πρώτα ποιο είναι το κύριο νομικό ζήτημα που θέτει το Question και έπειτα χρησιμοποίησε τα Facts για να κρατήσεις μόνο τα πραγματικά περιστατικά, τους νομικούς όρους και τις κρίσιμες έννοιες που βοηθούν στην ανάκτηση σχετικών νομικών κειμένων με βάση ακριβείς ή κοντινές λεκτικές αντιστοιχίες.

Η αναδιατύπωση πρέπει να είναι κατάλληλη για sparse retrieval, όπως BM25 ή keyword-based retrieval, ώστε η αναζήτηση να βασίζεται σε σημαντικούς νομικούς όρους, ουσιαστικά, ρήματα, θεσμούς, άρθρα, νόμους, δικαστικές αποφάσεις και κρίσιμες πραγματικές περιστάσεις που είναι πιθανό να εμφανίζονται στα σχετικά νομικά κείμενα.

Οδηγίες:
- Γράψε μόνο στα ελληνικά, με ελληνική νομική ορολογία.
- Μην απαντήσεις στο ερώτημα.
- Μην κάνεις νομική ανάλυση.
- Παράδωσε μόνο μία λίστα από περίπου 20 έως 30 λέξεις-κλειδιά ή σύντομες φράσεις-κλειδιά, χωρισμένες με κόμμα.
- Στηρίξου αποκλειστικά στα στοιχεία που δίνονται στο αρχικό ερώτημα και διατήρησε αυτούσια τυχόν συγκεκριμένα άρθρα, νόμους, αριθμούς αποφάσεων ή λοιπές νομικές παραπομπές που αναφέρονται ρητά.
- Παράδωσε μόνο γενικευμένους αλλά στοχευμένους νομικούς όρους και σύντομες φράσεις-κλειδιά που βοηθούν τη λεξική ανάκτηση, αποφεύγοντας πλήρεις προτάσεις, ανάλυση, αφηγηματική διατύπωση και υπερβολικά συγκεκριμένα πραγματικά στοιχεία όπως ονόματα, πόλεις ή τυχαίες λεπτομέρειες, εκτός αν είναι νομικά κρίσιμα ή αναφέρονται ως άρθρα, νόμοι ή αποφάσεις.

Παρακάτω υπάρχουν παραδείγματα για τον τρόπο με τον οποίο πρέπει να γίνει η αναδιατύπωση.

Παράδειγμα με τα ίδια πραγματικά περιστατικά αλλά διαφορετικά ερωτήματα:

Facts:

1. Η εταιρεία Α, με έδρα την Αθήνα, πώλησε στον Β, κάτοικο Πάτρας, επαγγελματικό εξοπλισμό για κατάστημα που ο Β λειτουργεί στη Λάρισα.
2. Η σύμβαση καταρτίστηκε ηλεκτρονικά, το τίμημα συμφωνήθηκε να καταβληθεί σε δύο δόσεις και η παράδοση του εξοπλισμού έγινε στη Λάρισα.
3. Μετά την παράδοση, ο Β ισχυρίστηκε ότι ο εξοπλισμός εμφάνιζε ουσιώδη ελαττώματα και αρνήθηκε να καταβάλει τη δεύτερη δόση του τιμήματος.
4. Ο Β διαθέτει επίσης εξοχική κατοικία στον Βόλο και αποθήκη στην Καλαμάτα, οι οποίες δεν συνδέονται με τη σύμβαση ή με την παράδοση του εξοπλισμού.

Question 1:
Ποιο δικαστήριο είναι κατά τόπον αρμόδιο για την αγωγή της εταιρείας Α κατά του Β σχετικά με την καταβολή του υπόλοιπου τιμήματος;

Καλή λίστα λέξεων-κλειδιών 1:
κατά τόπον αρμοδιότητα, τοπική αρμοδιότητα, αρμόδιο δικαστήριο, δωσιδικία, γενική δωσιδικία, ειδική δωσιδικία, συντρέχουσα δωσιδικία, δωσιδικία συμβατικών διαφορών, δωσιδικία σύμβασης, αγωγή από σύμβαση, αγωγή καταβολής τιμήματος, απαίτηση καταβολής τιμήματος, υπόλοιπο τιμήματος, συμβατική απαίτηση, συμβατική ενοχή, σύμβαση πώλησης, πώληση κινητού πράγματος, πώληση επαγγελματικού εξοπλισμού, πωλητής, αγοραστής, ενάγων πωλητής, εναγόμενος αγοραστής, έδρα νομικού προσώπου, κατοικία εναγομένου, τόπος κατάρτισης σύμβασης, ηλεκτρονική κατάρτιση σύμβασης, τόπος εκπλήρωσης παροχής, τόπος παράδοσης πράγματος, χρηματική παροχή, συμβατική διαφορά
Question 2:
Μπορεί ο Β να αρνηθεί την καταβολή της δεύτερης δόσης λόγω των ελαττωμάτων του εξοπλισμού και ποιες αξιώσεις μπορεί να προβάλει από την ελαττωματική πώληση;

Καλή λίστα λέξεων-κλειδιών 2:
ελαττωματική πώληση, πραγματικά ελαττώματα, ουσιώδη ελαττώματα, έλλειψη συμφωνημένων ιδιοτήτων, ευθύνη πωλητή, ευθύνη από πώληση, δικαιώματα αγοραστή, αξιώσεις αγοραστή, αγοραστής ελαττωματικού πράγματος, σύμβαση πώλησης, πώληση κινητού πράγματος, πώληση επαγγελματικού εξοπλισμού, παράδοση ελαττωματικού πράγματος, ελαττώματα μετά την παράδοση, πλημμελής εκπλήρωση, μη προσήκουσα εκπλήρωση, αντισυμβατική παροχή, άρνηση καταβολής τιμήματος, μη καταβολή τιμήματος, κατακράτηση τιμήματος, δόση τιμήματος, υπόλοιπο τιμήματος, ένσταση μη εκπλήρωσης, ένσταση μη εκπληρωθέντος συναλλάγματος, μείωση τιμήματος, υπαναχώρηση από πώληση, αποζημίωση λόγω ελαττώματος, συμβατική ευθύνη, προστασία αγοραστή

Στα παραδείγματα φαίνεται ότι τα ίδια Facts οδηγούν σε διαφορετική λίστα λέξεων-κλειδιών όταν αλλάζει το Question. Δώσε προτεραιότητα στο συγκεκριμένο νομικό ζήτημα του Question και χρησιμοποίησε μόνο τα Facts που το εξειδικεύουν ή επηρεάζουν τη νομική του κατανόηση. Παράλειψε δευτερεύοντα ή περιγραφικά στοιχεία που εμφανίζονται στην υπόθεση αλλά δεν βοηθούν τη sparse ανάκτηση για το συγκεκριμένο ερώτημα.

Τώρα δημιούργησε λίστα λέξεων-κλειδιών για το παρακάτω ερώτημα με τον ίδιο τρόπο:

{original_query}"""

DENSE_REWRITE_PROMPT = """Διάβασε το παρακάτω ελληνικό νομικό ερώτημα.

Ο ρόλος σου είναι να μετατρέψεις το αρχικό ερώτημα σε στοχευμένο ερώτημα ανάκτησης για dense retrieval σε ελληνικά νομικά κείμενα. Εντόπισε πρώτα ποιο είναι το κύριο νομικό ζήτημα που θέτει το Question και έπειτα χρησιμοποίησε τα Facts για να κρατήσεις μόνο τα πραγματικά περιστατικά που βοηθούν στην κατανόηση αυτού του ζητήματος. Η αναδιατύπωση πρέπει να είναι κατάλληλη για σημασιολογική αναζήτηση, ώστε το dense retrieval να μπορεί να συλλάβει καλύτερα το νόημα της υπόθεσης και να ανακτήσει σχετικά νομικά κείμενα.

Οδηγίες:
- Γράψε μόνο στα ελληνικά, με ελληνική νομική ορολογία.
- Μην απαντήσεις στο ερώτημα.
- Μην κάνεις νομική ανάλυση.
- Στηρίξου αποκλειστικά στα στοιχεία που δίνονται στο αρχικό ερώτημα και διατήρησε τυχόν συγκεκριμένα άρθρα, νόμους ή δικαστικές αποφάσεις που αναφέρονται ρητά.
- Παράδωσε μία ενιαία παράγραφο νομικής γλώσσας, ξεκινώντας απευθείας από το κρίσιμο νομικό ζήτημα και ενσωματώνοντας τις απαραίτητες έννοιες, σχέσεις και πραγματικά περιστατικά που βοηθούν τη σημασιολογική ανάκτηση.

Παρακάτω υπάρχουν παραδείγματα για τον τρόπο με τον οποίο πρέπει να γίνει η αναδιατύπωση.

Παράδειγμα με τα ίδια πραγματικά περιστατικά αλλά διαφορετικά ερωτήματα:

Facts:

1. Η εταιρεία Α, με έδρα την Αθήνα, πώλησε στον Β, κάτοικο Πάτρας, επαγγελματικό εξοπλισμό για κατάστημα που ο Β λειτουργεί στη Λάρισα.
2. Η σύμβαση καταρτίστηκε ηλεκτρονικά, το τίμημα συμφωνήθηκε να καταβληθεί σε δύο δόσεις και η παράδοση του εξοπλισμού έγινε στη Λάρισα.
3. Μετά την παράδοση, ο Β ισχυρίστηκε ότι ο εξοπλισμός εμφάνιζε ουσιώδη ελαττώματα και αρνήθηκε να καταβάλει τη δεύτερη δόση του τιμήματος.
4. Ο Β διαθέτει επίσης εξοχική κατοικία στον Βόλο και αποθήκη στην Καλαμάτα, οι οποίες δεν συνδέονται με τη σύμβαση ή με την παράδοση του εξοπλισμού.

Question 1:
Ποιο δικαστήριο είναι κατά τόπον αρμόδιο για την αγωγή της εταιρείας Α κατά του Β σχετικά με την καταβολή του υπόλοιπου τιμήματος;

Καλή αναδιατύπωση 1:
Κατά τόπον αρμοδιότητα δικαστηρίου για αγωγή πωλήτριας εταιρείας κατά αγοραστή σχετικά με καταβολή υπολοίπου τιμήματος από σύμβαση πώλησης επαγγελματικού εξοπλισμού. Κρίσιμες έννοιες είναι η έδρα της ενάγουσας εταιρείας, η κατοικία του εναγομένου, ο τόπος κατάρτισης της σύμβασης, ο τόπος παράδοσης ή εκπλήρωσης της παροχής και η σύνδεση της διαφοράς με τη συμβατική ενοχή. Παραλείπονται περιουσιακά στοιχεία του αγοραστή που δεν συνδέονται με την αρμοδιότητα.

Question 2:
Μπορεί ο Β να αρνηθεί την καταβολή της δεύτερης δόσης λόγω των ελαττωμάτων του εξοπλισμού και ποιες αξιώσεις μπορεί να προβάλει από την ελαττωματική πώληση;

Καλή αναδιατύπωση 2:
Δικαιώματα αγοραστή από ελαττωματική πώληση επαγγελματικού εξοπλισμού και δυνατότητα άρνησης καταβολής υπολοίπου τιμήματος λόγω ουσιωδών ελαττωμάτων μετά την παράδοση. Κρίσιμες έννοιες είναι η ευθύνη του πωλητή για πραγματικά ελαττώματα, η πλημμελής εκπλήρωση της σύμβασης πώλησης, η ένσταση μη εκπλήρωσης, η μείωση τιμήματος, η υπαναχώρηση ή αποζημίωση και η σχέση των ελαττωμάτων με την οφειλή της δεύτερης δόσης.

Στα παραδείγματα φαίνεται ότι τα ίδια Facts οδηγούν σε διαφορετική αναδιατύπωση όταν αλλάζει το Question. Δώσε προτεραιότητα στο συγκεκριμένο νομικό ζήτημα του Question και χρησιμοποίησε μόνο τα Facts που το εξειδικεύουν ή επηρεάζουν τη νομική του κατανόηση. Παράλειψε δευτερεύοντα ή περιγραφικά στοιχεία που εμφανίζονται στην υπόθεση αλλά δεν βοηθούν τη σημασιολογική ανάκτηση για το συγκεκριμένο ερώτημα.

Τώρα αναδιατύπωσε το παρακάτω ερώτημα με τον ίδιο τρόπο:

{original_query}"""


In [ ]:
CHAT_LOG = []
def chat(system, user, max_tokens=42000, temperature=0.2):
    started = time.perf_counter()
    response = post_json(CHAT_URL + "/chat/completions", {
        "model": CHAT_MODEL, "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "temperature": temperature, "top_p": 1.0, "max_tokens": max_tokens,
        "reasoning_effort": "low",
    }, CHAT_API_KEY)
    usage = response.get("usage") or {}
    CHAT_LOG.append({"seconds": time.perf_counter() - started,
                     "prompt_tokens": usage.get("prompt_tokens"),
                     "completion_tokens": usage.get("completion_tokens"),
                     "total_tokens": usage.get("total_tokens")})
    content = response["choices"][0]["message"]["content"] or ""
    return re.sub(r"<(?:think|analysis)>.*?</(?:think|analysis)>", "", content, flags=re.S | re.I).strip()

def rewrite_all(kind):
    path = OUTPUT_DIR / f"{kind}_rewrites.json"
    saved = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    prompt = SPARSE_REWRITE_PROMPT if kind == "sparse" else DENSE_REWRITE_PROMPT
    for qid, original in queries.items():
        if qid in saved: continue
        raw = chat("You are a Greek legal retrieval query rewriter. Follow the user's requested output format.", prompt.format(original_query=original))
        match = re.search(r"(?:KEYWORDS|REWRITTEN_QUERY|QUERY)\s*:\s*(.*)", raw, re.I | re.S)
        saved[qid] = clean(match.group(1) if match else raw)
        path.write_text(json.dumps(saved, ensure_ascii=False, indent=2), encoding="utf-8")
    return saved

sparse_rewrites = dense_rewrites = None
if RUN_REFORMULATION:
    sparse_rewrites = rewrite_all("sparse"); dense_rewrites = rewrite_all("dense")
    for name, index in sparse.items():
        register("Reform-" + name, {q: index.search(sparse_rewrites[q]) for q in queries})
    for name in MODELS:
        if name == "Gemini-001": continue  # paper reports Gemini only for vanilla Greek retrieval
        run, _, _ = dense_run(name, dense_rewrites)
        register("Reform-" + name, run)

## 6. PRF, fusion, BM25 tuning, and English translation

Sparse PRF: first-pass top 10, select 30 highest TF × IDF non-query terms, then search again. Dense PRF: top-10 centroid, `norm(1.0 × query + 0.25 × centroid)`. RRF combines BM25-GreekStemmer and each of the eight locally hosted dense models using their top 100 and `k=60`. The BM25 sweep is **diagnostic**: choosing parameters on all benchmark qrels gives an oracle upper bound, not a deployable tuned model. The English baseline translates the **Hugging Face** corpus and queries with `gpt-oss-120b` while retaining IDs and qrels; translations are cached per item. Translating the full corpus is expensive.

In [ ]:
TRANSLATION_SYSTEM = """You are a meticulous legal translator for Greek → English.
Translate the provided Greek legal text into faithful, formal legal English.

Hard requirements:
- Do NOT omit, summarize, paraphrase, simplify, modernize, or “improve” content. Preserve all information.
- Preserve the original legal facts, legal relationships, dates, procedural posture, parties, and statutory references exactly in meaning.
- Preserve structure: headings, numbering, article numbering, paragraphs, bullet points, indentation, and line breaks where possible.
- Preserve citations exactly: laws, articles, paragraphs, case numbers, FEK references, dates, court names, ECLI, and any other legal identifiers.
- Preserve quotes, brackets, parentheses, punctuation, and emphasis markers.
- Do not adapt Greek civil-law concepts into different common-law concepts if that would change the legal meaning.
- Translate legal terms consistently. If a Greek legal term is ambiguous, choose the most legally appropriate English term and keep it consistent.
- Keep Greek proper nouns for persons, places, and institutions in Latin transliteration if there is no standard English form; if a standard English form exists, use it. Optionally keep the Greek in parentheses on first mention.
- Do not infer missing information. Do not add commentary, notes, explanations, warnings, or legal analysis.
- Do not reveal reasoning, hidden analysis, chain-of-thought, or translation notes.
- Output ONLY the translated English text.

Formatting:
- Keep line breaks as in the source where possible.
- Do not wrap the translation in markdown fences.
- If the input contains multiple distinct parts, translate them in the same order without reordering."""

TRANSLATION_USER = """Translate the following Greek legal text into faithful English.

Text type: {text_type}
Item id: {item_id}

Return ONLY the English translation of the text between the delimiters.

<<<BEGIN_GREEK_TEXT
{source_text}
END_GREEK_TEXT>>>"""


In [ ]:
def sparse_prf(index, text, feedback_k=10, expansion_n=30):
    original = index.tokenize(text)
    first = index.search_tokens(original, feedback_k)
    positions = {d: i for i, d in enumerate(index.ids)}
    tf = Counter(t for d in first for t in index.tokens[positions[d]])
    terms = sorted((t for t in tf if t not in set(original) and index.index.idf.get(t, 0) > 0),
                   key=lambda t: (-tf[t] * index.index.idf[t], t))[:expansion_n]
    return index.search_tokens(original + terms)

def dense_prf_run(name, D, Q, ids=doc_ids, qids=query_ids):
    out = {}
    for i, qid in enumerate(qids):
        first = np.argsort(-(D @ Q[i]), kind="stable")[:10]
        centroid = D[first].mean(axis=0)
        vector = Q[i] + 0.25 * centroid
        vector /= max(np.linalg.norm(vector), 1e-12)
        out[qid] = top_ids(D @ vector, ids)
    return out

def rrf(left, right, k=60):
    out = {}
    for qid in queries:
        scores = Counter()
        for rank, docid in enumerate(left[qid][:100], 1): scores[docid] += 1 / (k + rank)
        for rank, docid in enumerate(right[qid][:100], 1): scores[docid] += 1 / (k + rank)
        out[qid] = [d for d, _ in sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:100]]
    return out

if RUN_PRF_RRF_TUNING:
    base = sparse["BM25-GreekStemmer"]
    register("PRF-BM25", {q: sparse_prf(base, text) for q, text in queries.items()})
    for name in ("Qwen3-8B", "EmbGemma-300M"):
        D, Q = dense_vectors[name]
        register("PRF-" + name, dense_prf_run(name, D, Q))
    for name in MODELS:
        if name == "Gemini-001": continue
        register("RRF-BM25+" + name, rrf(runs["BM25-GreekStemmer"], runs[name]))
    sweep = []
    for query_mode, texts in [("original", queries), ("reform", sparse_rewrites)]:
        if texts is None: continue
        for k1 in sorted(set([1.5] + list(np.round(np.arange(0.2, 3.01, 0.2), 2)))):
            for b in sorted(set([0.75] + list(np.round(np.arange(0, 1.01, 0.1), 2)))):
                idx = BM25Okapi(base.tokens, k1=float(k1), b=float(b))
                ranking = {q: top_ids(idx.get_scores(base.tokenize(texts[q])), base.ids, positive_only=True) for q in queries}
                recall = np.mean([score_ranking(ranking[q], qrels[q])["Recall@100"] for q in queries])
                sweep.append((query_mode, k1, b, recall, ranking))
    for mode in {x[0] for x in sweep}:
        best = max((x for x in sweep if x[0] == mode), key=lambda x: x[3])
        register(f"Oracle-tuned-BM25-{mode}-k1{best[1]}-b{best[2]}", best[4])

if RUN_TRANSLATION:
    def translate_map(items, text_type):
        path = OUTPUT_DIR / f"english_{text_type}.json"
        translated = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
        for item_id, source_text in items.items():
            if item_id in translated: continue
            prompt = TRANSLATION_USER.format(text_type=text_type, item_id=item_id, source_text=source_text)
            translated[item_id] = chat(TRANSLATION_SYSTEM, prompt, max_tokens=32768, temperature=0.0)
            path.write_text(json.dumps(translated, ensure_ascii=False, indent=2), encoding="utf-8")
        return {item_id: translated[item_id] for item_id in items}
    en_docs = translate_map(docs, "article")
    en_queries = translate_map(queries, "query")
    assert set(en_docs) == set(docs) and set(en_queries) == set(queries)
    en_index = SparseIndex(en_docs, make_spacy_tokens("en"))
    register("English-BM25-spaCy", {q: en_index.search(en_queries[q]) for q in queries})
    for name in MODELS:
        if name == "Gemini-001": continue
        ranking, _, _ = dense_run(name, en_queries, en_docs, "en")
        register("English-" + name, ranking)

## 7. ReAct-BM25 (the reported ten-round method)

Each round: (1) planner writes a Greek legal keyword query; (2) BM25-GreekStemmer retrieves up to 100 articles; (3) observer reads **the complete article texts** and returns binary keep decisions. The next planner prompt receives the original question, prior queries, and up to 27 previously kept full texts. Final ranking favors **number of rounds kept**, then **earliest kept round**, then **best BM25 rank**. It contains only observer-kept articles, capped at 100. Save rankings after rounds 1, 2, 3, 5, 10; round 0 is vanilla BM25.

The original Greek planner and observer prompts are embedded below. The notebook splits observer batches by token budget without truncating an article. A run can require roughly 20 LLM calls per query, more if batches are needed; use `QUERY_LIMIT` to inspect a small subset before a full run.

In [ ]:
PLANNER_SYSTEM = """Είσαι βοηθός για sparse retrieval σε ελληνικά νομικά κείμενα.

Σκοπός σου είναι να δημιουργείς BM25 keyword queries με ελληνική νομική ορολογία.
Δεν απαντάς στο νομικό ερώτημα. Δεν κάνεις πλήρη νομική ανάλυση.
Δεν προσπαθείς να μαντέψεις ποιος νόμος, ποιο άρθρο ή ποια απόφαση εφαρμόζεται.
Βοηθάς μόνο στην ανάκτηση σχετικών νομικών κειμένων με βάση το ερώτημα του χρήστη.

Σε κάθε απάντηση πρέπει να δίνεις ακριβώς δύο ενότητες:

REASON:
Σύντομη εξήγηση, στα ελληνικά, για το ποιο είναι το κύριο νομικό ζήτημα που στοχεύει το query
και γιατί οι επιλεγμένοι όροι είναι χρήσιμοι για BM25 retrieval.
Μην γράφεις τελική νομική απάντηση.
Μην παρουσιάζεις ως βέβαιο ότι εφαρμόζεται συγκεκριμένος νόμος ή άρθρο, εκτός αν αναφέρεται ρητά στο ερώτημα.

QUERY:
Λέξεις-κλειδιά και σύντομες νομικές φράσεις στα ελληνικά, χωρισμένες με κόμμα.
Χρησιμοποίησε όρους που είναι πιθανό να εμφανίζονται σε νόμους, αποφάσεις, δικόγραφα,
νομική θεωρία ή δικαστική γλώσσα.

Κανόνες:
- Το QUERY δεν πρέπει να είναι πρόταση ή απάντηση.
- Το QUERY πρέπει να περιέχει μόνο όρους αναζήτησης.
- Κράτησε αυτούσιες τυχόν ρητές παραπομπές σε άρθρα, νόμους ή αποφάσεις.
- Μην επινοείς άρθρα, νόμους, αποφάσεις ή document IDs.
- Μην μαντεύεις συγκεκριμένο εφαρμοστέο νόμο, άρθρο ή απόφαση αν δεν υπάρχει ρητή ένδειξη στο ερώτημα."""


In [ ]:
PLANNER_FIRST = """Διάβασε το παρακάτω ελληνικό νομικό ερώτημα.

Ο ρόλος σου είναι να μετατρέψεις το αρχικό ερώτημα σε στοχευμένο BM25 keyword query για sparse retrieval σε ελληνικά νομικά κείμενα.

Εντόπισε πρώτα το κύριο νομικό ζήτημα που θέτει το Question και χρησιμοποίησε τα Facts μόνο στον βαθμό που βοηθούν την ανάκτηση σχετικών νομικών κειμένων.

Το QUERY πρέπει να βασίζεται σε όρους που είναι πιθανό να εμφανίζονται σε νόμους, δικονομικά κείμενα, νομική θεωρία ή δικαστική γλώσσα: νομικές έννοιες, αξιώσεις, ενστάσεις, προϋποθέσεις, ένδικα βοηθήματα, έννομες συνέπειες, άρθρα, νόμους ή αποφάσεις όταν αναφέρονται ρητά.

Οδηγίες:
- Γράψε στα ελληνικά, με ελληνική νομική ορολογία.
- Δώσε περίπου 20 έως 30 λέξεις-κλειδιά ή σύντομες φράσεις-κλειδιά, χωρισμένες με κόμμα.
- Κράτησε αυτούσια τυχόν άρθρα, νόμους, αριθμούς αποφάσεων ή άλλες νομικές παραπομπές που αναφέρονται ρητά.

Παράδειγμα με τα ίδια Facts αλλά διαφορετικά Questions:

Facts:
1. Η εταιρεία Α πώλησε στον Β επαγγελματικό εξοπλισμό.
2. Το τίμημα συμφωνήθηκε να καταβληθεί σε δύο δόσεις.
3. Μετά την παράδοση, ο Β ισχυρίστηκε ότι ο εξοπλισμός εμφάνιζε ουσιώδη ελαττώματα και αρνήθηκε να καταβάλει τη δεύτερη δόση.

Question 1:
Ποιο δικαστήριο είναι κατά τόπον αρμόδιο για την αγωγή της εταιρείας Α κατά του Β σχετικά με την καταβολή του υπόλοιπου τιμήματος;

Καλό QUERY 1:
κατά τόπον αρμοδιότητα, τοπική αρμοδιότητα, αρμόδιο δικαστήριο, δωσιδικία, γενική δωσιδικία, ειδική δωσιδικία, δωσιδικία συμβατικών διαφορών, αγωγή από σύμβαση, αγωγή καταβολής τιμήματος, απαίτηση καταβολής τιμήματος, υπόλοιπο τιμήματος, συμβατική απαίτηση, συμβατική ενοχή, σύμβαση πώλησης, πωλητής, αγοραστής, κατοικία εναγομένου, τόπος κατάρτισης σύμβασης, τόπος εκπλήρωσης παροχής, χρηματική παροχή, συμβατική διαφορά

Question 2:
Μπορεί ο Β να αρνηθεί την καταβολή της δεύτερης δόσης λόγω των ελαττωμάτων του εξοπλισμού και ποιες αξιώσεις μπορεί να προβάλει από την ελαττωματική πώληση;

Καλό QUERY 2:
ελαττωματική πώληση, πραγματικά ελαττώματα, ουσιώδη ελαττώματα, έλλειψη συμφωνημένων ιδιοτήτων, ευθύνη πωλητή, δικαιώματα αγοραστή, αξιώσεις αγοραστή, σύμβαση πώλησης, παράδοση ελαττωματικού πράγματος, πλημμελής εκπλήρωση, μη προσήκουσα εκπλήρωση, άρνηση καταβολής τιμήματος, κατακράτηση τιμήματος, δόση τιμήματος, υπόλοιπο τιμήματος, ένσταση μη εκπλήρωσης, ένσταση μη εκπληρωθέντος συναλλάγματος, μείωση τιμήματος, υπαναχώρηση από πώληση, αποζημίωση λόγω ελαττώματος, συμβατική ευθύνη, προστασία αγοραστή

Στα παραδείγματα φαίνεται ότι τα ίδια Facts οδηγούν σε διαφορετικό QUERY όταν αλλάζει το Question. Δώσε προτεραιότητα στο συγκεκριμένο νομικό ζήτημα του Question.

Αρχικό νομικό ερώτημα:

{original_query}

Δημιούργησε το πρώτο BM25 query.

Χρησιμοποίησε ακριβώς τη μορφή:

REASON:
<σύντομη εξήγηση γιατί αυτά τα keywords ταιριάζουν στο κύριο νομικό ζήτημα>

QUERY:
<20 έως 30 ελληνικές νομικές λέξεις-κλειδιά ή σύντομες φράσεις, χωρισμένες με κόμμα>"""


In [ ]:
PLANNER_NEXT = """Αρχικό νομικό ερώτημα:

{original_query}

Γύρος:
{round_idx} από {max_rounds}

Προηγούμενα BM25 queries:
{query_history}

Πλήρη κείμενα από έγγραφα που έχουν ήδη σημειωθεί ως σχετικά:
{kept_docs_summary}

Δημιούργησε το επόμενο BM25 query.

Σκοπός αυτού του γύρου είναι να καλύψεις διαφορετική νομική πτυχή του αρχικού ερωτήματος
ή να δοκιμάσεις εναλλακτική ελληνική νομική ορολογία για κάτι που μπορεί να μην ανακτήθηκε καλά.

Μην επαναλάβεις απλώς το ίδιο query, εκτός αν αυτό είναι αναγκαίο.
Προτίμησε όρους που είναι πιθανό να εμφανίζονται σε νόμους, αποφάσεις, δικόγραφα,
νομική θεωρία ή δικαστική γλώσσα.

Χρησιμοποίησε ακριβώς τη μορφή:

REASON:
<σύντομη εξήγηση ποια νέα retrieval πτυχή στοχεύει αυτό το query>

QUERY:
<20 έως 30 ελληνικές νομικές λέξεις-κλειδιά ή σύντομες φράσεις, χωρισμένες με κόμμα>"""


In [ ]:
OBSERVER_SYSTEM = """Είσαι κριτής ανάκτησης για ελληνικά νομικά κείμενα.

Θα σου δοθεί ένα αρχικό νομικό ερώτημα και πολλά αποτελέσματα BM25.
Η δουλειά σου είναι μόνο να επιλέξεις ποια από τα δοσμένα έγγραφα είναι χρήσιμα ως πηγές για την απάντηση.

Ένα έγγραφο είναι RELEVANT όταν, διαβάζοντας το αρχικό νομικό ερώτημα, το περιεχόμενό του φαίνεται ότι θα μπορούσε να χρησιμοποιηθεί ως πηγή για τη σύνθεση της απάντησης. Δεν χρειάζεται το έγγραφο να απαντά μόνο του ολόκληρο το ερώτημα. Αρκεί να καλύπτει ένα ουσιαστικό μέρος του ζητήματος, να δίνει χρήσιμο νομικό πλαίσιο ή να περιέχει πληροφορία που θα χρειαζόταν κάποιος για να απαντήσει σωστά.

Επειδή εξετάζεις πολλά αποτελέσματα BM25 μαζί, μην περιοριστείς στα πιο προφανή ή στα πρώτα σχετικά έγγραφα. Να σκέφτεσαι για κάθε έγγραφο χωριστά αν θα άξιζε να δοθεί στον τελικό απαντητή ως υλικό τεκμηρίωσης. Αν η πληροφορία του εγγράφου μπορεί πράγματι να βοηθήσει στην απάντηση, κράτησέ το ως RELEVANT, ακόμα και αν υπάρχουν άλλα έγγραφα που φαίνονται πιο σημαντικά.

Για κάθε RELEVANT έγγραφο, γράψε το doc_id και έναν σύντομο λόγο στα ελληνικά.

Χρησιμοποίησε ακριβώς αυτή τη μορφή:

RELEVANT:
<doc_id> | <σύντομος λόγος στα ελληνικά>

Αν κανένα έγγραφο δεν είναι relevant, γράψε ακριβώς:

RELEVANT: NONE

Κανόνες:
- Μην απαντάς στο νομικό ερώτημα.
- Μην επινοείς document IDs.
- Χρησιμοποίησε μόνο document IDs από τα δοσμένα BM25 αποτελέσματα.
- Μην γράψεις NOT_RELEVANT.
- Μην γράψεις MISSING."""


In [ ]:
OBSERVER_USER = """Αρχικό νομικό ερώτημα:

{original_query}

Τρέχοντα BM25 αποτελέσματα:

{retrieved_docs_block}

Εξέτασε μόνο τα παραπάνω έγγραφα.

Διάβασε τα αποτελέσματα ως υποψήφιες πηγές για την απάντηση στο αρχικό νομικό ερώτημα. Μην προσπαθήσεις απλώς να διαλέξεις τα λίγα καλύτερα έγγραφα. Εξέτασε κάθε έγγραφο χωριστά και κράτησε όσα μπορούν πράγματι να χρησιμοποιηθούν για τη σύνθεση της απάντησης ή για ένα αναγκαίο μέρος της.

Ένα έγγραφο πρέπει να σημειωθεί ως RELEVANT όταν προσφέρει χρήσιμη νομική πληροφορία για το ερώτημα, ακόμα και αν δεν απαντά μόνο του ολόκληρο το ερώτημα. Αγνόησε έγγραφα που έχουν μόνο κοινές λέξεις ή γενική θεματική σχέση με το ερώτημα, χωρίς να προσφέρουν πραγματικά χρήσιμη πληροφορία για την απάντηση.

Χρησιμοποίησε μόνο την εξής μορφή:

RELEVANT:
<doc_id> | <σύντομος λόγος στα ελληνικά>

Αν κανένα από τα τρέχοντα αποτελέσματα δεν είναι relevant, γράψε ακριβώς:

RELEVANT: NONE

Μην γράψεις NOT_RELEVANT.
Μην γράψεις MISSING.
Μην απαντήσεις στο νομικό ερώτημα.
Μην χρησιμοποιήσεις document IDs που δεν υπάρχουν στα BM25 αποτελέσματα."""


In [ ]:
def parse_planner(raw):
    raw = re.sub(r"<(?:think|analysis)>.*?</(?:think|analysis)>", "", raw, flags=re.S | re.I)
    parts = re.split(r"(?im)^\s*(?:\*\*)?QUERY(?:\*\*)?\s*:\s*", raw, maxsplit=1)
    return clean(parts[-1] if len(parts) > 1 else raw)

def parse_observer(raw, valid):
    raw = re.sub(r"<(?:think|analysis)>.*?</(?:think|analysis)>", "", raw, flags=re.S | re.I)
    raw = re.split(r"(?im)^\s*(?:NOT_RELEVANT|MISSING|OBSERVATION|ANSWER)\s*:", raw)[0]
    keep = []
    for line in raw.splitlines():
        line = re.sub(r"^\s*[-*•]+\s*", "", line.strip())
        docid = clean(line.split("|", 1)[0])
        if docid in valid and docid not in keep: keep.append(docid)
    return keep

def observer_batches(original, retrieved, max_input_tokens=84000):
    try:
        import tiktoken
        try: enc = tiktoken.get_encoding("o200k_harmony")
        except KeyError: enc = tiktoken.get_encoding("o200k_base")
        count = lambda s: len(enc.encode(s, disallowed_special=())) + 64
    except ImportError:
        count = lambda s: len(s.encode("utf-8")) + 64  # conservative upper bound
    batches, current = [], []
    for item in retrieved:
        candidate = current + [item]
        block = "\n\n".join(f"[DOC]\ndoc_id: {d}\ntext: {docs[d]}\n[/DOC]" for d in candidate)
        prompt = OBSERVER_SYSTEM + OBSERVER_USER.format(original_query=original, retrieved_docs_block=block)
        if count(prompt) < max_input_tokens:
            current = candidate
        elif current:
            batches.append(current); current = [item]
            one = OBSERVER_SYSTEM + OBSERVER_USER.format(original_query=original, retrieved_docs_block=f"[DOC]\ndoc_id: {item}\ntext: {docs[item]}\n[/DOC]")
            if count(one) >= max_input_tokens: raise RuntimeError(f"Article {item} exceeds observer context")
        else:
            raise RuntimeError(f"Article {item} exceeds observer context")
    if current: batches.append(current)
    return batches

def react_rank(kept):
    return sorted(kept, key=lambda d: (-len(kept[d]["rounds"]), kept[d]["first"], kept[d]["best_rank"], d))[:100]

def react_one(qid, original, index, rounds=10):
    history, candidates, kept, snapshots, trace = [], set(), {}, {}, []
    for round_number in range(1, rounds + 1):
        call_start = len(CHAT_LOG)
        if round_number == 1:
            prompt = PLANNER_FIRST.format(original_query=original, max_rounds=rounds)
        else:
            earlier = "\n".join(f"- Γύρος {i}: {query}" for i, query in history)
            ranked_kept = react_rank(kept)[:27]
            kept_text = "\n\n".join(f"[ΣΧΕΤΙΚΟ_ΚΕΙΜΕΝΟ]\n{docs[d]}\n[/ΣΧΕΤΙΚΟ_ΚΕΙΜΕΝΟ]" for d in ranked_kept) or "Δεν έχουν σημειωθεί ακόμη σχετικά έγγραφα."
            prompt = PLANNER_NEXT.format(original_query=original, round_idx=round_number, max_rounds=rounds,
                                         query_history=earlier, kept_docs_summary=kept_text)
        planner_raw = chat(PLANNER_SYSTEM, prompt)
        keyword_query = parse_planner(planner_raw) or original
        retrieved = index.search(keyword_query, 100, positive_only=True)
        history.append((round_number, keyword_query)); candidates.update(retrieved)
        rank_by_doc = {d: i for i, d in enumerate(retrieved, 1)}
        selected, observer_raw = [], []
        for batch in observer_batches(original, retrieved):
            block = "\n\n".join(f"[DOC]\ndoc_id: {d}\ntext: {docs[d]}\n[/DOC]" for d in batch)
            response = chat(OBSERVER_SYSTEM, OBSERVER_USER.format(original_query=original, retrieved_docs_block=block), max_tokens=44096)
            observer_raw.append(response)
            selected.extend(parse_observer(response, set(batch)))
        for d in dict.fromkeys(selected):
            if d not in kept: kept[d] = {"rounds": [], "first": round_number, "best_rank": rank_by_doc[d]}
            kept[d]["rounds"].append(round_number)
            kept[d]["best_rank"] = min(kept[d]["best_rank"], rank_by_doc[d])
        snapshots[round_number] = react_rank(kept)
        trace.append({"round": round_number, "query": keyword_query, "planner_raw": planner_raw,
                      "observer_raw_batches": observer_raw, "llm_calls": CHAT_LOG[call_start:], "retrieved": retrieved,
                      "observer_kept": list(dict.fromkeys(selected)), "ranked_kept": snapshots[round_number],
                      "candidate_recall": len(candidates & set(qrels[qid])) / len(qrels[qid])})
    return snapshots, trace

if RUN_REACT:
    index = sparse["BM25-GreekStemmer"]
    stage_runs = {n: {} for n in (1, 2, 3, 5, 10)}
    traces = {}
    trace_path = OUTPUT_DIR / "ReAct-BM25-traces.json"
    if trace_path.exists(): traces = json.loads(trace_path.read_text(encoding="utf-8"))
    for qid, original in queries.items():
        if qid not in traces:
            snapshots, trace = react_one(qid, original, index)
            traces[qid] = {"snapshots": snapshots, "trace": trace}
            trace_path.write_text(json.dumps(traces, ensure_ascii=False, indent=2), encoding="utf-8")
        for n in stage_runs:
            stage_runs[n][qid] = traces[qid]["snapshots"][str(n)] if isinstance(next(iter(traces[qid]["snapshots"])), str) else traces[qid]["snapshots"][n]
    for n, run in stage_runs.items(): register(f"ReAct-BM25-round-{n}", run)
    register("ReAct-BM25", stage_runs[10])
    rows = []
    for n in stage_runs:
        rows.append({"round": n,
                     "candidate_pool_Recall@100": np.mean([traces[q]["trace"][n-1]["candidate_recall"] for q in queries]),
                     "observer_kept_Recall@100": np.mean([score_ranking(stage_runs[n][q], qrels[q])["Recall@100"] for q in queries]),
                     "calls/query": np.mean([sum(len(t["llm_calls"]) for t in traces[q]["trace"][:n]) for q in queries]),
                     "tokens/query": np.mean([sum((c.get("total_tokens") or 0) for t in traces[q]["trace"][:n] for c in t["llm_calls"]) for q in queries])})
    display(pd.DataFrame(rows).round(3))

## 8. Results and reproducibility

This table is computed from the current run; it is **not** a transcription of paper numbers. The paper's main reported Recall@100 values are BM25 0.36, Reform-BM25 0.60, Qwen3-8B 0.67, Reform-Qwen3-8B 0.73, Gemini-001 0.77, and ten-round ReAct-BM25 0.67. Further differences can come from model versions, local serving, stochastic generation, or preprocessing versions. Do not present a `QUERY_LIMIT` run as a full-dataset aggregate. ReAct candidate-pool recall is diagnostic; final retrieval metrics always use observer-kept rankings.

Methods and settings follow [the paper](https://arxiv.org/html/2608.18752v2), especially Sections 3–4 and Appendices C–G. The Greek prompts and algorithm code are embedded here, so the notebook does not need the private exploratory scripts.

In [ ]:
results = pd.DataFrame(summaries)
if not results.empty:
    results = results[["experiment", "nDCG@10", "nDCG@100", "Recall@10", "Recall@100", "MAP@100"]]
    results.to_csv(OUTPUT_DIR / "paper_experiment_summary.csv", index=False)
    display(results.round(3))
else:
    print("No experiment switch has been enabled.")
print("Run artifacts:", OUTPUT_DIR)

### Optional paired randomization tests

The paper uses 100,000 random paired label swaps and Holm correction for 20 planned tests (five comparisons × four metrics). This cell repeats that analysis on whichever query set was loaded above. It requires the named runs to have been generated. Leave `RUN_SIGNIFICANCE=False` for routine exploration.

In [ ]:
RUN_SIGNIFICANCE = False
if RUN_SIGNIFICANCE:
    comparisons = [
        ("Reform-BM25-GreekStemmer", "BM25-GreekStemmer"),
        ("Reform-Qwen3-8B", "Qwen3-8B"),
        ("ReAct-BM25", "Reform-BM25-GreekStemmer"),
        ("ReAct-BM25", "Reform-Qwen3-8B"),
        ("ReAct-BM25", "Gemini-001"),
    ]
    metrics = ["nDCG@10", "nDCG@100", "Recall@100", "MAP@100"]
    rng = np.random.default_rng(42)
    tests = []
    for a, b in comparisons:
        if a not in runs or b not in runs: raise ValueError(f"Run {a} and {b} first")
        for metric in metrics:
            diff = np.asarray([score_ranking(runs[a][q], qrels[q])[metric] - score_ranking(runs[b][q], qrels[q])[metric] for q in queries])
            observed = abs(diff.mean())
            extreme = 0
            for _ in range(100):  # batches of 1,000 keep memory bounded
                signs = rng.choice([-1, 1], size=(1000, len(diff)))
                extreme += np.count_nonzero(abs((signs * diff).mean(axis=1)) >= observed - 1e-12)
            p = (extreme + 1) / 100001
            tests.append({"comparison": f"{a} vs {b}", "metric": metric, "delta": diff.mean(), "p_raw": p})
    ordered = sorted(range(len(tests)), key=lambda i: tests[i]["p_raw"])
    adjusted = [0.0] * len(tests); running = 0.0
    for rank, i in enumerate(ordered):
        running = max(running, min(1.0, (len(tests) - rank) * tests[i]["p_raw"]))
        adjusted[i] = running
    for row, p in zip(tests, adjusted): row["p_holm"] = p
    display(pd.DataFrame(tests).round(4))